# Baseline Forecasting Models and Rolling Evaluation

This notebook implements simple baseline forecasting models to serve as reference points for more advanced approaches.

The models are evaluated using a rolling-origin strategy that mimics real-time forecasting conditions.


In [2]:
import sys
from pathlib import Path
import pandas as pd

#make src importable 
PROJECT_ROOT = Path.cwd().parent   # notebooks/ -> project/
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_project_paths
from src.io_data import read_ari_ili
from src.gaps import gap_summary

#  load data 
paths = get_project_paths(PROJECT_ROOT)
ari_df, ili_df = read_ari_ili(paths.data)

# (optional) quick check
ari_df.head(), ili_df.head()
from src.impute import impute_series_weekly
from src.baselines import rolling_baselines, summarize_baselines
from src.calendars import make_calendars_from_df, extract_raw_weekly_series

In [3]:
gap_ari = gap_summary(ari_df, disease="ARI", calendar_mode="data_range")
gap_ili = gap_summary(ili_df, disease="ILI", calendar_mode="data_range")

MIN_OBS = 300
MAX_GAP = 8

ok_ari = gap_ari[(gap_ari["n_observed"] >= MIN_OBS) & (gap_ari["max_gap_length"] <= MAX_GAP)]["location"].tolist()
ok_ili = gap_ili[(gap_ili["n_observed"] >= MIN_OBS) & (gap_ili["max_gap_length"] <= MAX_GAP)]["location"].tolist()

ok_ari, ok_ili


(['BE', 'BG', 'CZ', 'DE', 'EE', 'LT', 'RO', 'SI'],
 ['BE', 'CZ', 'EE', 'GR', 'IE', 'LT', 'NL', 'PL', 'RO', 'SI'])

## Baseline models

The following baseline forecasting strategies are considered:

- **Naive last observation**: forecasts are equal to the most recent observed value
- **Naive seasonal**: forecasts correspond to the same week of the previous year
- **Mixed naive strategy**: short horizons use persistence, longer horizons use seasonal repetition

These baselines provide a minimum performance benchmark that any advanced model should outperform.


In [4]:
import numpy as np
import pandas as pd

from src.impute import impute_series_weekly
from src.metrics import mae, rmse, mase_scale_seasonal, mase_from_mae

# 1) Rolling baseline evaluator
def rolling_baselines(history_init: pd.Series, y_test: pd.Series, horizons=(1,2,3,4)):
    """
    history_init: training series (imputed) up to train_end
    y_test: truth series for 2023 (may contain NaNs)
    Simulates real-time availability:
      - each week in 2023 is "revealed" and appended to history if observed
      - then forecasts h=1..4 are produced
    """
    history = history_init.copy()
    results = []

    for origin in y_test.index:
        # reveal truth at origin (if observed)
        if pd.notna(y_test.loc[origin]):
            history.loc[origin] = float(y_test.loc[origin])

        # if still missing at origin, naive-last can't be computed
        if origin not in history.index or pd.isna(history.loc[origin]):
            continue

        for h in horizons:
            target = origin + pd.Timedelta(weeks=h)

            # SOLO comprobamos que el target existe en el calendario
            if target not in y_test.index:
                continue

            y_true = y_test.loc[target]

            pred_last = float(history.loc[origin])

            # seasonal naive: same week last year
            ref = target - pd.Timedelta(weeks=52)
            pred_year = float(history.loc[ref]) if (ref in history.index and pd.notna(history.loc[ref])) else np.nan

            # mixed: h=1,2 -> last; h=3,4 -> year
            pred_mix = pred_last if h in (1, 2) else pred_year

            results.append({
                "origin": origin,
                "target": target,
                "h": h,
                "y": float(y_true) if pd.notna(y_true) else np.nan,
                "pred_last": pred_last,
                "pred_year": pred_year,
                "pred_mix": pred_mix
            })

    return pd.DataFrame(results)


# 2) Per-country baselines + MAE/RMSE/MASE
def run_country_baselines(df: pd.DataFrame, location: str, disease: str,
                          train_end="2022-12-25",
                          test_start="2023-01-01", test_end="2023-12-31",
                          m_season=52):
    """
    Computes rolling-origin baseline metrics for one country and returns a tidy table:
    disease, location, h, model, MAE, RMSE, MASE, n
    Classic MASE uses training-only seasonal naive scale with period m=52.
    """
    df = df.copy()
    df["truth_date"] = pd.to_datetime(df["truth_date"])

    # Global calendar (common across countries inside this df)
    full_weeks = pd.date_range(
        start=pd.Timestamp("2014-01-05"),
        end=df["truth_date"].max(),
        freq="W-SUN"
    )

    train_end = pd.Timestamp(train_end)
    test_calendar = full_weeks[(full_weeks >= pd.Timestamp(test_start)) & (full_weeks <= pd.Timestamp(test_end))]
    train_calendar = full_weeks[full_weeks <= train_end]

    #  training (imputed) 
    train_imputed = impute_series_weekly(
        df[df["location"] == location][["truth_date", "value"]],
        calendar=train_calendar,
        trim_to_first_obs=True,
        max_gap_knn=8
    )

    # MASE scale computed ONLY on training 
    scale = mase_scale_seasonal(train_imputed, m=m_season)

    # raw truth series for 2023 (no imputing) 
    raw_full = (
        df[df["location"] == location]
        .sort_values("truth_date")
        .drop_duplicates("truth_date", keep="last")
        .set_index("truth_date")["value"]
        .reindex(full_weeks)
    )
    y_test = raw_full.reindex(test_calendar)

    #  rolling evaluation 
    df_eval = rolling_baselines(train_imputed, y_test, horizons=(1,2,3,4))

    #  metrics per horizon & model 
    rows = []
    model_cols = [("naive_last", "pred_last"),
                  ("naive_year", "pred_year"),
                  ("naive_mixed", "pred_mix")]

    for h in (1, 2, 3, 4):
        sub = df_eval[df_eval["h"] == h]

        for model_name, col in model_cols:
            sub_m = sub[sub["y"].notna()].copy()
            sub_m = sub_m[sub_m[col].notna()]
            if len(sub_m) == 0:
                continue

            m_mae = mae(sub_m["y"].values, sub_m[col].values)
            m_rmse = rmse(sub_m["y"].values, sub_m[col].values)
            m_mase = mase_from_mae(m_mae, scale)

            rows.append({
                "disease": disease,
                "location": location,
                "h": h,
                "model": model_name,
                "MAE": m_mae,
                "RMSE": m_rmse,
                "MASE": m_mase,
                "n": int(len(sub_m))
            })

    return pd.DataFrame(rows)


# 3) Run for all selected countries (ARI/ILI)

baselines_ari_list = []
for loc in ok_ari:
    try:
        baselines_ari_list.append(run_country_baselines(ari_df, loc, "ARI"))
    except Exception as e:
        print(f"Skipping ARI {loc}: {e}")

baselines_ari = pd.concat(baselines_ari_list, ignore_index=True) if baselines_ari_list else pd.DataFrame()
baselines_ari.head()


baselines_ili_list = []
for loc in ok_ili:
    try:
        baselines_ili_list.append(run_country_baselines(ili_df, loc, "ILI"))
    except Exception as e:
        print(f"Skipping ILI {loc}: {e}")

baselines_ili = pd.concat(baselines_ili_list, ignore_index=True) if baselines_ili_list else pd.DataFrame()
baselines_ili.head()


# 4) mean across countries view

baselines_ari.groupby(["h","model"])[["MAE","RMSE","MASE"]].mean()
baselines_ili.groupby(["h","model"])[["MAE","RMSE","MASE"]].mean()


MAE        RMSE      MASE
h model                                       
1 naive_last   31.296410   53.223860  0.585357
  naive_mixed  31.296410   53.223860  0.585357
  naive_year   87.454850  126.129768  1.575618
2 naive_last   42.662637   68.240651  0.795409
  naive_mixed  42.662637   68.240651  0.795409
  naive_year   81.918720  115.909189  1.475545
3 naive_last   54.158768   84.020045  0.963901
  naive_mixed  78.443499  110.114593  1.392860
  naive_year   78.443499  110.114593  1.392860
4 naive_last   61.648671   93.271916  1.101897
  naive_mixed  75.652539  106.487761  1.345440
  naive_year   75.652539  106.487761  1.345440

In [5]:
from src.config import get_project_paths

PROJECT_ROOT = Path.cwd().parent
paths = get_project_paths(PROJECT_ROOT)

(paths.results).mkdir(parents=True, exist_ok=True)

baselines_ari.to_csv(paths.results / "baselines_ari_rolling_2023.csv", index=False)
baselines_ili.to_csv(paths.results / "baselines_ili_rolling_2023.csv", index=False)

print("Saved:",
      paths.results / "baselines_ari_rolling_2023.csv",
      "and",
      paths.results / "baselines_ili_rolling_2023.csv")


Saved: C:\Users\aolas\UNI PYTHON\TFG\results\baselines_ari_rolling_2023.csv and C:\Users\aolas\UNI PYTHON\TFG\results\baselines_ili_rolling_2023.csv
